# FAST-UAV - Multirotor Design Optimization
*Author: Félix Pollet - 2023* <br>

This notebook is basically the same as the [Tutorial Notebook](0_Tutorial.ipynb), but it is more concise so that you can quickly re-use it to find the optimum sizing for any multirotor UAV specification.

## 1. Setting up and analyzing the problem

Now that you are familiar with FAST-UAV, you may define your own design problem. In the following cell, feel free to use your own source file (initially, [problem_inputs_DJI_M600.xml](../data/problem_inputs_DJI_M600.xml)) and configuration file (initially, [multirotor_mdo.yaml](../configurations/multirotor_mdo.yaml)). Remember that the configuration file defines the design problem, i.e. the model, the optimization problem definition and the optimization algorithm, while the source file is a reference file for the problem's input values.

In [1]:
# Import required librairies
import os.path as pth
import openmdao.api as om
import logging
import warnings
import shutil
import fastoad.api as oad
from time import time
import matplotlib.pyplot as plt
from fastuav.utils.postprocessing.analysis_and_plots import *

# Declare paths to folders and files
DATA_FOLDER_PATH = "./data"
WORK_FOLDER_PATH = "./workdir"
CONFIGURATION_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "configurations")
SOURCE_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "source_files")

CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "multirotor_mdo.yaml") # You may provide a different configuration file
SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_DJI_M600.xml") # You may provide a different source file

# For having log messages display on screen
#logging.basicConfig(level=logging.INFO, format="%(levelname)-8s: %(message)s")
#warnings.filterwarnings(action="ignore")

# For using all screen width
from IPython.display import display, HTML, IFrame
display(HTML("<style>.container { width:95% !important; }</style>"))

c:\Users\user\.conda\envs\FastUAV\lib\site-packages\stdatm\__init__.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Let's have a look at the [N2 diagram](http://openmdao.org/twodocs/versions/latest/basic_guide/make_n2.html) to see the structure of the model:

In [2]:
N2_FILE = pth.join(WORK_FOLDER_PATH, "n2.html")
oad.write_n2(CONFIGURATION_FILE, N2_FILE, overwrite=True)
from IPython.display import IFrame
IFrame(src=N2_FILE, width="100%", height="500px")

Now we can generate the temporary input file `workdir/problem_inputs.xml` from the source file you have specified earlier:

In [4]:
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)

'D:/KangDH/fast_UAV/FAST-UAV/src/fastuav/notebooks/workdir/problem_inputs.xml'

이제 생성된 [입력 파일](./workdir/problem_inputs.xml)을 체크아웃할 수 있습니다. 이 파일의 값은 사용자가 수정할 수 있습니다(예: 원본 소스 파일에 지정된 값이 정확하지 않거나 누락된 경우). 이러한 변경 사항은 이 노트북의 범위에 유지됩니다. 즉, 소스 파일에는 영향을 미치지 않으며 세션을 나가면 변경 사항이 사라집니다.

In [5]:
INPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml")
oad.variable_viewer(INPUT_FILE)

## 2. Running the MDO

You can now run an optimization problem. Remember that the last part of the configuration file .yaml is where this optimization problem is defined. For example, you may change the objective function to maximize the range instead of minimizing the mtow.

```yaml
optimization:
  design_variables:
    - name: data:weights:mtow:k # over estimation coefficient on the load mass
      upper: 40.0
      lower: 1.0
  constraints:
    - name: optimization:constraints:weights:mtow:guess # mass consistency
      lower: 0.0
  objective:
    - name: data:weights:mtow
      scaler: 1e-1
```

In [6]:
optim_problem = oad.optimize_problem(CONFIGURATION_FILE, overwrite=True)

Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.5423052396301786
            Iterations: 8
            Function evaluations: 8
            Gradient evaluations: 7
Optimization Complete
-----------------------------------


Let's save these results permanently:

In [7]:
OUTPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_outputs.xml")
DJI_M600_OUTPUT_FILE = pth.join(SOURCE_FOLDER_PATH, 'problem_outputs_DJI_M600_mdo.xml')
shutil.copy(OUTPUT_FILE, DJI_M600_OUTPUT_FILE)

'./data\\source_files\\problem_outputs_DJI_M600_mdo.xml'

The `optimizer_viewer` offers a convenient summary of the optimization result. If design variables or constraints have active bounds they are yellow whereas they are red if they are violated.

In [8]:
oad.optimization_viewer(CONFIGURATION_FILE)

c:\Users\user\.conda\envs\FastUAV\lib\site-packages\jupyter_client\session.py:718: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


You can use the `VariableViewer` tool to see the optimization results for all variables of the system by loading the .xml output file:

In [9]:
oad.variable_viewer(OUTPUT_FILE)

## 3. Analysis and plots

You can now use postprocessing plots to visualize the results of the MDO.

In [10]:
fig = multirotor_geometry_plot(DJI_M600_OUTPUT_FILE)
fig.show()

In [11]:
fig = mass_breakdown_sun_plot_drone(DJI_M600_OUTPUT_FILE)
fig.show()